### HOMEWORK - EVALUATION

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
len(documents)

72

In [7]:
doc = documents[0]
print(doc["filename"])
print(doc["content"])

01-agentic-rag/lessons/01-intro.md
# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple la

In [8]:
doc

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

#### Q1. Generating questions

In [6]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [9]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
import json

user_prompt = json.dumps(doc)

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [13]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [15]:
result = response.output_parsed

print(result.questions)

['What exactly is retrieval-augmented generation, and why does it help with wrong or outdated answers?', 'Why do you use the model like a black box instead of explaining how it works internally?', 'How does this course plan to build the FAQ assistant, and what will it answer from?', 'What will be covered in the first part of the module before the agentic version starts?', 'What changes in the second part when the system becomes more agent-like?']


In [21]:
from evaluation_utils import llm_structured, calc_price, llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/02-environment.md
- 01-agentic-rag/lessons/03-rag.md

In [31]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents[:3], generate_ground_truth)

  0%|          | 0/3 [00:00<?, ?it/s]

In [32]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

15

In [34]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=117, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1137),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=124, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1410),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=1280), output_tokens=91, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1844)]

In [38]:
(usages[0].input_tokens + usages[1].input_tokens + usages[2].input_tokens) / 3

1353.0

The average of input tokens is 1353

Answer Q1: 1400

In [39]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [40]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [41]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)